## Import

In [1]:
import os
import re
import torch
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# 환경 설정
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Using device: cuda


## Fixed RandomSeed

In [2]:
# 시드 고정
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

## Load Pre-trained Model

In [3]:
# 모델 로딩
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    device_map="auto",
    torch_dtype=torch.float16
)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards: 100%|██████████| 2/2 [00:22<00:00, 11.27s/it]


In [4]:
# model parameter 수 검색
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

    # print("Model parameter count:", count_parameters(model))
print("Model parameter count:", count_parameters(model))

Model parameter count: 3744761856


## Inference

In [ ]:
# 정답 알파벳 추출 함수
def extract_answer_letter(text):
    match = re.search(r"Answer:\s*([A-Da-d])\b", text)
    return match.group(1).upper() if match else "?"

In [9]:
# 추론
test = pd.read_csv('/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/test.csv')
image_dir = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/'
results = []

for _, row in tqdm(test.iterrows(), total=len(test)):
    # print(f"Processing image: {row['img_path']}")
    # image = Image.open(row['img_path']).convert("RGB")
    image_path = os.path.join(image_dir, row['img_path'][2:])
    if not os.path.exists(image_path):
        print(f"Image not found: {image_path}")
        continue
    image = Image.open(image_path).convert("RGB")
    choices = [row[c] for c in ['A', 'B', 'C', 'D']]

    prompt = (
        "You are a helpful AI that answers multiple-choice questions based on the given image.\n"
        "Select the best answer from A, B, C, or D.\n\n"
        f"Question: {row['Question']}\n"
        + "\n".join([f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)]) +
        "\nAnswer:"
    )

    inputs = processor(images=image, text=prompt, return_tensors="pt")
    inputs = {k: (v.half().to(device) if v.dtype == torch.float32 else v.to(device)) for k, v in inputs.items()}

    output = model.generate(**inputs, max_new_tokens=3, do_sample=False, temperature=0.0)
    decoded = processor.tokenizer.decode(output[0], skip_special_tokens=True).strip()
    results.append(extract_answer_letter(decoded))
print('✅ Done.')

  0%|          | 1/680 [00:01<17:36,  1.56s/it]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  0%|          | 3/680 [00:01<05:13,  2.16it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  1%|          | 5/680 [00:01<02:58,  3.79it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  1%|          | 7/680 [00:02<02:03,  5.44it/s]The foll

✅ Done.


## Submission

In [11]:
submission = pd.read_csv('./sample_submission.csv')
submission['answer'] = results  
submission.to_csv('./baseline_submit.csv', index=False)
print("✅ Done.")

✅ Done.
